# OCTMNIST revisado: oito arquiteturas com progresso compacto

Este notebook substitui os experimentos comparativos dos baselines. Ele corrige os pontos metodológicos destacados pelos pareceres:

- todas as arquiteturas são treinadas **do zero** (`weights=None` e `pretrained=False`);
- CrossEntropyLoss é o protocolo principal;
- todas usam as mesmas divisões oficiais, transformações e critério de checkpoint;
- o melhor checkpoint é selecionado por **macro-AUC de validação**;
- a tabela principal contém somente métricas de **teste**;
- cada modelo é avaliado com:
  1. distribuição natural;
  2. CWS;
  3. class-balanced sampling;
  4. class-weighted CrossEntropy;
- são registradas cinco sementes, custos, parâmetros, históricos e matrizes;
- a busca de hiperparâmetros utiliza o mesmo orçamento para todas as arquiteturas.

Os oito modelos deste notebook, somados ao MedLDLMO-CWS do primeiro notebook, completam as nove arquiteturas do artigo.


In [1]:
%pip install -q --upgrade-strategy only-if-needed numpy pandas scipy scikit-learn matplotlib opencv-python-headless albumentations medmnist timm torch torchvision pillow psutil "jinja2>=3.1.5"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import copy
import gc
import itertools
import html
import json
import os
import random
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import medmnist
import numpy as np
import pandas as pd
import timm
import psutil
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from medmnist import INFO
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import models as tv_models
from IPython.display import HTML, display

warnings.filterwarnings("ignore")


class CompactTrainingMonitor:
    """Painel HTML que atualiza o mesmo output, sem criar uma linha por época."""

    def __init__(self):
        self.handle = None

    @staticmethod
    def _metric(value, digits=4):
        if value is None:
            return "aguardando"
        try:
            value = float(value)
            if not np.isfinite(value):
                return "aguardando"
            return f"{value:.{digits}f}"
        except (TypeError, ValueError):
            return str(value)

    @staticmethod
    def _bar(percent, label):
        percent = max(0.0, min(100.0, float(percent)))
        return f"""
        <div style="margin:4px 0 8px 0;">
          <div style="display:flex;justify-content:space-between;font-size:12px;">
            <span>{html.escape(label)}</span><strong>{percent:.1f}%</strong>
          </div>
          <div style="height:11px;background:#e5e7eb;border-radius:7px;overflow:hidden;">
            <div style="height:11px;width:{percent:.2f}%;background:#2563eb;"></div>
          </div>
        </div>
        """

    def update(
        self,
        *,
        phase,
        model_name,
        strategy,
        seed,
        run_index,
        run_total,
        epoch,
        max_epochs,
        status="Treinando",
        batch_index=None,
        batch_total=None,
        train_loss=None,
        val_loss=None,
        val_auc=None,
        best_auc=None,
        best_epoch=None,
        no_improvement=0,
        patience=0,
        start_time=None,
        final_metrics=None,
    ):
        max_epochs = max(1, int(max_epochs))
        run_total = max(1, int(run_total))
        epoch = max(0, int(epoch))

        if batch_index is not None and batch_total:
            batch_fraction = max(0.0, min(1.0, batch_index / max(1, batch_total)))
            epoch_fraction = min(1.0, ((max(epoch, 1) - 1) + batch_fraction) / max_epochs)
            batch_text = f"Lote {batch_index}/{batch_total}"
        else:
            epoch_fraction = min(1.0, epoch / max_epochs)
            batch_text = ""

        overall_fraction = min(
            1.0,
            ((max(1, run_index) - 1) + epoch_fraction) / run_total,
        )

        elapsed = 0.0 if start_time is None else max(0.0, time.perf_counter() - start_time)
        elapsed_text = time.strftime("%H:%M:%S", time.gmtime(elapsed))

        ram_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
        gpu_text = "CPU"
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
            gpu_text = f"GPU {allocated:.2f}/{reserved:.2f} GB"

        safe_phase = html.escape(str(phase))
        safe_model = html.escape(str(model_name))
        safe_strategy = html.escape(str(strategy))
        safe_status = html.escape(str(status))

        final_html = ""
        if final_metrics:
            final_html = (
                "<div style='margin-top:8px;padding:7px;background:#ecfdf5;border-radius:6px;'>"
                f"<strong>Resultado:</strong> ACC={self._metric(final_metrics.get('accuracy'))} | "
                f"AUC={self._metric(final_metrics.get('macro_auc'))} | "
                f"Macro-F1={self._metric(final_metrics.get('macro_f1'))}"
                "</div>"
            )

        card = f"""
        <div style="font-family:Arial,sans-serif;border:1px solid #d1d5db;border-radius:9px;
                    padding:12px;max-width:950px;background:#ffffff;color:#111827;">
          <div style="display:flex;justify-content:space-between;gap:12px;align-items:center;">
            <div>
              <strong>{safe_phase}</strong>
              <span style="color:#6b7280;"> | execução {run_index}/{run_total}</span>
            </div>
            <div style="font-size:12px;color:#4b5563;">{safe_status} | {elapsed_text}</div>
          </div>

          {self._bar(overall_fraction * 100, "Progresso geral")}

          <div style="font-size:14px;margin-bottom:4px;">
            <strong>{safe_model}</strong> | estratégia={safe_strategy} | seed={seed}
          </div>

          {self._bar(epoch_fraction * 100, f"Época {epoch}/{max_epochs} {batch_text}".strip())}

          <div style="display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));
                      gap:6px;font-size:12px;margin-top:8px;">
            <div><strong>Train loss</strong><br>{self._metric(train_loss)}</div>
            <div><strong>Val loss</strong><br>{self._metric(val_loss)}</div>
            <div><strong>Val AUC</strong><br>{self._metric(val_auc)}</div>
            <div><strong>Melhor AUC</strong><br>{self._metric(best_auc)}
                {f" (ép. {best_epoch})" if best_epoch not in (None, -1) else ""}</div>
          </div>

          <div style="margin-top:8px;font-size:12px;color:#4b5563;">
            Early stopping: {no_improvement}/{patience} |
            RAM: {ram_gb:.2f} GB | {gpu_text}
          </div>
          {final_html}
        </div>
        """

        if self.handle is None:
            self.handle = display(HTML(card), display_id=True)
        else:
            self.handle.update(HTML(card))


TRAINING_MONITOR = CompactTrainingMonitor()


## 1. Configuração

Rode primeiro em `smoke`. Depois altere para `final`.

O modo final executa oito modelos × quatro estratégias × cinco sementes. É uma bateria extensa. O arquivo `raw_results.csv` permite retomada automática.


In [3]:
MODE = "final"  # "smoke" ou "final"
assert MODE in {"smoke", "final"}

CLASS_NAMES = ["CNV", "DME", "Drusen", "Normal"]
NUM_CLASSES = 4
DRUSEN_CLASS = 2
NORMAL_CLASS = 3
MINORITY_CLASSES = (1, 2)
CWS_N_CLUSTERS = 3
CWS_SEED = 2026

RESULTS_DIR = Path(f"results_baselines_revised_{MODE}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for folder in ["predictions", "histories", "confusion_matrices", "figures"]:
    (RESULTS_DIR / folder).mkdir(exist_ok=True)

if MODE == "smoke":
    SEEDS = [42]
    MAX_EPOCHS = 2
    PATIENCE = 2
    TUNING_EPOCHS = 1
    TUNING_FRACTION = 0.02
    MODELS_TO_RUN = ["ResNet18", "MedConvMixer-LT"]
    STRATEGIES_TO_RUN = ["natural", "cws"]
    RUN_HYPERPARAMETER_TUNING = False
else:
    SEEDS = [42, 123, 456, 789, 2026]
    MAX_EPOCHS = 50
    PATIENCE = 7
    TUNING_EPOCHS = 3
    TUNING_FRACTION = 1.0
    MODELS_TO_RUN = [
        "ResNet18",
        "DenseNet121",
        "EfficientNet-B0-CBAM-LT",
        "MedConvMixer-LT",
        "TinyViT-LT",
        "MedMobileViT-LT",
        "MaxViT-LT",
        "Swin-Transformer-LT",
    ]
    STRATEGIES_TO_RUN = [
        "natural",
        "cws",
        "balanced_sampler",
        "class_weighted",
    ]
    RUN_HYPERPARAMETER_TUNING = True

NUM_WORKERS = 2
DASHBOARD_UPDATE_SECONDS = 15
SHOW_FIGURES = MODE == "smoke"
MIN_DELTA = 1e-4

# Estabilidade numérica
MAX_GRAD_NORM = 1.0
EVALUATION_USE_AMP = False
RETRY_WITHOUT_AMP_ON_NAN = True
DROPOUT = 0.3
DEFAULT_LR = 1e-3
DEFAULT_WEIGHT_DECAY = 1e-4

TUNING_LEARNING_RATES = [3e-4, 7e-4, 1e-3]
TUNING_WEIGHT_DECAYS = [1e-5, 1e-4]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = device.type == "cuda"

print("Modo:", MODE)
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Modo: final
Dispositivo: cuda
GPU: NVIDIA GeForce RTX 5090


In [4]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


seed_everything(SEEDS[0])


## 2. Dataset e CWS

O CWS usa somente imagens do treino para criar subgrupos de DME e Drusen. As transformações são iguais para todas as classes. MaxViT e Swin recebem redimensionamento bicúbico para 224×224; os demais modelos usam 28×28.


In [5]:
def build_train_transform(image_size: int) -> A.Compose:
    transforms = []
    if image_size != 28:
        transforms.append(
            A.Resize(image_size, image_size, interpolation=cv2.INTER_CUBIC)
        )
    transforms.extend([
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.90, 1.10),
            translate_percent=(-0.05, 0.05),
            rotate=(-10, 10),
            shear=0,
            p=0.30,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.10,
            contrast_limit=0.10,
            p=0.20,
        ),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])
    return A.Compose(transforms)


def build_eval_transform(image_size: int) -> A.Compose:
    transforms = []
    if image_size != 28:
        transforms.append(
            A.Resize(image_size, image_size, interpolation=cv2.INTER_CUBIC)
        )
    transforms.extend([
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])
    return A.Compose(transforms)


info = INFO["octmnist"]
DataClass = getattr(medmnist, info["python_class"])
train_raw = DataClass(split="train", download=True)
val_raw = DataClass(split="val", download=True)
test_raw = DataClass(split="test", download=True)

TRAIN_TARGETS = np.asarray(train_raw.labels).reshape(-1).astype(int)
VAL_TARGETS = np.asarray(val_raw.labels).reshape(-1).astype(int)
TEST_TARGETS = np.asarray(test_raw.labels).reshape(-1).astype(int)


class IndexedOCTDataset(Dataset):
    def __init__(self, raw_dataset, transform: Optional[A.Compose] = None):
        self.raw_dataset = raw_dataset
        self.transform = transform

    def __len__(self):
        return len(self.raw_dataset)

    def __getitem__(self, index: int):
        image, label = self.raw_dataset[index]
        image = np.asarray(image)
        if image.ndim == 2:
            image = image[..., None]
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        label = int(np.asarray(label).reshape(-1)[0])
        return image.float(), torch.tensor(label, dtype=torch.long), index


_DATASET_CACHE: Dict[int, Tuple[Dataset, Dataset, Dataset]] = {}


def get_datasets(image_size: int):
    if image_size not in _DATASET_CACHE:
        _DATASET_CACHE[image_size] = (
            IndexedOCTDataset(train_raw, build_train_transform(image_size)),
            IndexedOCTDataset(val_raw, build_eval_transform(image_size)),
            IndexedOCTDataset(test_raw, build_eval_transform(image_size)),
        )
    return _DATASET_CACHE[image_size]


In [6]:
def build_cws_groups(
    raw_images: np.ndarray,
    targets: np.ndarray,
    n_clusters: int = CWS_N_CLUSTERS,
    seed: int = CWS_SEED,
):
    groups = np.full(len(targets), -1, dtype=int)
    summary = []

    for class_id in MINORITY_CLASSES:
        indices = np.flatnonzero(targets == class_id)
        x = np.asarray(raw_images)[indices].reshape(len(indices), -1).astype(np.float32) / 255.0

        reducer = PCA(
            n_components=min(32, x.shape[1], len(indices) - 1),
            svd_solver="randomized",
            random_state=seed + class_id,
        )
        embedding = reducer.fit_transform(x)
        clusterer = MiniBatchKMeans(
            n_clusters=n_clusters,
            n_init=10,
            batch_size=1024,
            random_state=seed + class_id,
        )
        labels = clusterer.fit_predict(embedding)
        groups[indices] = labels

        for subgroup, count in enumerate(np.bincount(labels, minlength=n_clusters)):
            summary.append({
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "subgroup": subgroup,
                "count": int(count),
            })

    return groups, pd.DataFrame(summary)


CWS_GROUPS, cws_summary = build_cws_groups(train_raw.imgs, TRAIN_TARGETS)
display(cws_summary)
cws_summary.to_csv(RESULTS_DIR / "cws_group_summary.csv", index=False)


,class_id,class_name,subgroup,count
0,1,DME,0,1886
1,1,DME,1,7092
2,1,DME,2,1235
3,2,Drusen,0,5524
4,2,Drusen,1,1095
5,2,Drusen,2,1135


In [7]:
def build_sampling_weights(
    targets: np.ndarray,
    groups: np.ndarray,
    use_cws: bool,
    balanced: bool,
) -> Optional[np.ndarray]:
    if not use_cws and not balanced:
        return None

    counts = np.bincount(targets, minlength=NUM_CLASSES).astype(float)
    class_masses = (
        np.full(NUM_CLASSES, 1 / NUM_CLASSES)
        if balanced
        else counts / counts.sum()
    )
    weights = np.zeros(len(targets), dtype=np.float64)

    for class_id in range(NUM_CLASSES):
        class_indices = np.flatnonzero(targets == class_id)
        class_mass = class_masses[class_id]

        if use_cws and class_id in MINORITY_CLASSES:
            subgroup_values = np.unique(groups[class_indices])
            subgroup_values = subgroup_values[subgroup_values >= 0]
            for subgroup in subgroup_values:
                subgroup_indices = class_indices[groups[class_indices] == subgroup]
                weights[subgroup_indices] = (
                    class_mass / len(subgroup_values) / len(subgroup_indices)
                )
        else:
            weights[class_indices] = class_mass / len(class_indices)

    return weights / weights.sum()


def class_loss_weights() -> torch.Tensor:
    counts = np.bincount(TRAIN_TARGETS, minlength=NUM_CLASSES).astype(float)
    weights = len(TRAIN_TARGETS) / (NUM_CLASSES * counts)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_loaders(
    image_size: int,
    seed: int,
    physical_batch_size: int,
    strategy: str,
    train_indices: Optional[np.ndarray] = None,
):
    use_cws = strategy == "cws"
    balanced = strategy == "balanced_sampler"

    train_dataset, val_dataset, test_dataset = get_datasets(image_size)
    if train_indices is None:
        active_dataset = train_dataset
        active_targets = TRAIN_TARGETS
        active_groups = CWS_GROUPS
    else:
        train_indices = np.asarray(train_indices)
        active_dataset = Subset(train_dataset, train_indices.tolist())
        active_targets = TRAIN_TARGETS[train_indices]
        active_groups = CWS_GROUPS[train_indices]

    sample_weights = build_sampling_weights(
        active_targets,
        active_groups,
        use_cws=use_cws,
        balanced=balanced,
    )
    generator = make_generator(seed)

    loader_kwargs = dict(
        batch_size=physical_batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        worker_init_fn=seed_worker,
        persistent_workers=NUM_WORKERS > 0,
    )

    if sample_weights is None:
        train_loader = DataLoader(
            active_dataset,
            shuffle=True,
            generator=generator,
            drop_last=True,
            **loader_kwargs,
        )
    else:
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
            generator=generator,
        )
        train_loader = DataLoader(
            active_dataset,
            sampler=sampler,
            drop_last=True,
            **loader_kwargs,
        )

    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


## 3. Definições das oito arquiteturas

As implementações customizadas são preservadas, mas a inicialização pré-treinada de MaxViT e Swin foi removida.


In [8]:
class ConvMixerPatchEmbedding(nn.Module):
    def __init__(self, in_channels=1, dim=64, patch_size=2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x):
        return self.proj(x)


class ConvMixerBlock(nn.Module):
    def __init__(self, dim=64, kernel_size=5):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                dim, dim, kernel_size, groups=dim, padding="same"
            ),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x) + x)


class MedConvMixerLT(nn.Module):
    def __init__(self, dim=64, depth=4, num_classes=NUM_CLASSES):
        super().__init__()
        self.patch_embed = ConvMixerPatchEmbedding(1, dim, 2)
        self.blocks = nn.Sequential(
            *[ConvMixerBlock(dim, kernel_size=5) for _ in range(depth)]
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.blocks(self.patch_embed(x))
        return self.fc(self.dropout(torch.flatten(self.pool(x), 1)))


In [9]:
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.SiLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        batch, channels, _, _ = x.shape
        scale = self.fc(self.pool(x).view(batch, channels)).view(batch, channels, 1, 1)
        return x * scale


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.shared = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.shared(self.avg(x)) + self.shared(self.max(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(
            2, 1, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        maximum = torch.max(x, dim=1, keepdim=True).values
        return self.sigmoid(self.conv(torch.cat([avg, maximum], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention()

    def forward(self, x):
        x = x * self.channel(x)
        return x * self.spatial(x)


class MBConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=4, stride=1):
        super().__init__()
        hidden = in_channels * expansion
        self.use_residual = stride == 1 and in_channels == out_channels
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.SiLU(),
            nn.Conv2d(
                hidden,
                hidden,
                3,
                stride=stride,
                padding=1,
                groups=hidden,
                bias=False,
            ),
            nn.BatchNorm2d(hidden),
            nn.SiLU(),
            SEBlock(hidden),
            nn.Conv2d(hidden, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out


class EfficientNetB0CBAMLT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.block1, self.cbam1 = MBConvBlock(32, 32), CBAM(32)
        self.block2, self.cbam2 = MBConvBlock(32, 64, stride=2), CBAM(64)
        self.block3, self.cbam3 = MBConvBlock(64, 128, stride=2), CBAM(128)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cbam1(self.block1(self.stem(x)))
        x = self.cbam2(self.block2(x))
        x = self.cbam3(self.block3(x))
        return self.fc(self.dropout(torch.flatten(self.pool(x), 1)))


In [10]:
class TinyPatchEmbedding(nn.Module):
    def __init__(self, in_channels=1, embed_dim=96):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim, 3, stride=2, padding=1),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
        )

    def forward(self, x):
        return self.proj(x)


class TinyBlock(nn.Module):
    def __init__(self, dim=96, heads=4, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attn(normalized, normalized, normalized)
        x = x + attended
        return x + self.mlp(self.norm2(x))


class TinyViTLT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=96, depth=4):
        super().__init__()
        self.patch = TinyPatchEmbedding(1, embed_dim)
        self.blocks = nn.Sequential(
            *[TinyBlock(embed_dim) for _ in range(depth)]
        )
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.blocks(x).mean(dim=1)
        return self.fc(self.dropout(x))


In [11]:
class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                kernel_size // 2,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=4, stride=1):
        super().__init__()
        hidden = in_channels * expansion
        self.use_residual = stride == 1 and in_channels == out_channels
        self.block = nn.Sequential(
            ConvBNAct(in_channels, hidden, kernel_size=1),
            ConvBNAct(hidden, hidden, kernel_size=3, stride=stride, groups=hidden),
            nn.Conv2d(hidden, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out


class TransformerEncoder(nn.Module):
    def __init__(self, dim, heads=4, mlp_dim=128, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attn(normalized, normalized, normalized)
        x = x + attended
        return x + self.mlp(self.norm2(x))


class MobileViTBlock(nn.Module):
    def __init__(self, in_channels, transformer_dim, depth=2):
        super().__init__()
        self.local_rep = nn.Sequential(
            ConvBNAct(in_channels, in_channels, kernel_size=3),
            ConvBNAct(in_channels, transformer_dim, kernel_size=1),
        )
        self.transformers = nn.Sequential(
            *[TransformerEncoder(transformer_dim) for _ in range(depth)]
        )
        self.fusion = ConvBNAct(transformer_dim, in_channels, kernel_size=1)

    def forward(self, x):
        y = self.local_rep(x)
        batch, channels, height, width = y.shape
        y = y.flatten(2).transpose(1, 2)
        y = self.transformers(y)
        y = y.transpose(1, 2).reshape(batch, channels, height, width)
        return x + self.fusion(y)


class MedMobileViTLT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = ConvBNAct(1, 16)
        self.block1 = InvertedResidual(16, 32)
        self.block2 = InvertedResidual(32, 64, stride=2)
        self.mobilevit1 = MobileViTBlock(64, 96)
        self.block3 = InvertedResidual(64, 96, stride=2)
        self.mobilevit2 = MobileViTBlock(96, 128)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(96, num_classes)

    def forward(self, x):
        x = self.block2(self.block1(self.stem(x)))
        x = self.mobilevit1(x)
        x = self.mobilevit2(self.block3(x))
        return self.fc(self.dropout(torch.flatten(self.pool(x), 1)))


In [12]:
def build_resnet18():
    model = tv_models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    model.fc = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(model.fc.in_features, NUM_CLASSES),
    )
    return model


def build_densenet121():
    model = tv_models.densenet121(weights=None)
    model.features.conv0 = nn.Conv2d(
        1, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.features.pool0 = nn.Identity()
    model.classifier = nn.Sequential(
        nn.Dropout(DROPOUT),
        nn.Linear(model.classifier.in_features, NUM_CLASSES),
    )
    return model


def build_maxvit():
    return timm.create_model(
        "maxvit_tiny_tf_224",
        pretrained=False,
        in_chans=1,
        num_classes=NUM_CLASSES,
        drop_rate=DROPOUT,
    )


def build_swin():
    return timm.create_model(
        "swin_tiny_patch4_window7_224",
        pretrained=False,
        in_chans=1,
        num_classes=NUM_CLASSES,
        drop_rate=DROPOUT,
    )


MODEL_SPECS = {
    "ResNet18": {
        "builder": build_resnet18,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "DenseNet121": {
        "builder": build_densenet121,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "EfficientNet-B0-CBAM-LT": {
        "builder": EfficientNetB0CBAMLT,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "MedConvMixer-LT": {
        "builder": MedConvMixerLT,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "TinyViT-LT": {
        "builder": TinyViTLT,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "MedMobileViT-LT": {
        "builder": MedMobileViTLT,
        "input_size": 28,
        "physical_batch_size": 32,
        "accumulation_steps": 1,
    },
    "MaxViT-LT": {
        "builder": build_maxvit,
        "input_size": 224,
        "physical_batch_size": 16,
        "accumulation_steps": 2,
    },
    "Swin-Transformer-LT": {
        "builder": build_swin,
        "input_size": 224,
        "physical_batch_size": 16,
        "accumulation_steps": 2,
    },
}


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for name in MODELS_TO_RUN:
    model = MODEL_SPECS[name]["builder"]().to(device)
    size = MODEL_SPECS[name]["input_size"]
    with torch.no_grad():
        output = model(torch.randn(1, 1, size, size, device=device))
    print(name, tuple(output.shape), f"{count_parameters(model):,} parâmetros")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ResNet18 (1, 4) 11,169,732 parâmetros
DenseNet121 (1, 4) 6,949,124 parâmetros
EfficientNet-B0-CBAM-LT (1, 4) 133,066 parâmetros
MedConvMixer-LT (1, 4) 25,028 parâmetros
TinyViT-LT (1, 4) 448,900 parâmetros
MedMobileViT-LT (1, 4) 544,884 parâmetros
MaxViT-LT (1, 4) 30,404,428 parâmetros
Swin-Transformer-LT (1, 4) 27,519,358 parâmetros


## 4. Métricas e função de treinamento unificada


In [13]:
def compute_macro_auc(
    y_true: np.ndarray,
    y_prob: np.ndarray,
) -> float:
    """Calcula Macro-AUC depois de validar as probabilidades."""
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    if y_prob.ndim != 2:
        raise FloatingPointError(
            f"y_prob deveria ser 2D, mas possui shape={y_prob.shape}."
        )

    if y_prob.shape != (len(y_true), NUM_CLASSES):
        raise FloatingPointError(
            "Dimensão incompatível entre rótulos e probabilidades: "
            f"y_true={y_true.shape}, y_prob={y_prob.shape}."
        )

    finite_mask = np.isfinite(y_prob)
    if not finite_mask.all():
        invalid_count = int((~finite_mask).sum())
        raise FloatingPointError(
            "Probabilidades de validação contêm NaN ou infinito: "
            f"{invalid_count} valores inválidos."
        )

    # Corrige apenas pequenos erros de arredondamento.
    y_prob = np.clip(y_prob, 0.0, 1.0)
    row_sums = y_prob.sum(axis=1, keepdims=True)

    if np.any(row_sums <= 0):
        raise FloatingPointError(
            "Há linhas de probabilidade cuja soma é zero."
        )

    y_prob = y_prob / row_sums
    y_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))

    return float(
        roc_auc_score(
            y_bin,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    )


def collect_predictions(
    model,
    loader,
    *,
    use_amp: bool = EVALUATION_USE_AMP,
):
    """Coleta predições em float32 por padrão."""
    model.eval()
    indices_all, true_all, prob_all = [], [], []

    with torch.no_grad():
        for batch_number, (images, labels, indices) in enumerate(
            loader,
            start=1,
        ):
            images = images.to(device, non_blocking=True)

            with autocast(
                device_type=device.type,
                enabled=bool(use_amp and AMP_ENABLED),
            ):
                logits = model(images)

            # Softmax em float32, fora do autocast.
            logits = logits.float()

            if not torch.isfinite(logits).all():
                invalid_count = int((~torch.isfinite(logits)).sum().item())
                raise FloatingPointError(
                    "Logits contêm NaN ou infinito durante avaliação. "
                    f"Lote={batch_number}, inválidos={invalid_count}."
                )

            probabilities = torch.softmax(logits, dim=1)

            if not torch.isfinite(probabilities).all():
                invalid_count = int(
                    (~torch.isfinite(probabilities)).sum().item()
                )
                raise FloatingPointError(
                    "Softmax produziu NaN ou infinito durante avaliação. "
                    f"Lote={batch_number}, inválidos={invalid_count}."
                )

            indices_all.extend(np.asarray(indices))
            true_all.extend(labels.numpy())
            prob_all.extend(probabilities.cpu().numpy())

    y_true = np.asarray(true_all, dtype=int)
    y_prob = np.asarray(prob_all, dtype=np.float64)

    return (
        np.asarray(indices_all),
        y_true,
        y_prob,
        y_prob.argmax(axis=1),
    )


def compute_metrics(y_true, y_prob):
    y_pred = y_prob.argmax(axis=1)
    per_precision, per_recall, per_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_auc": compute_macro_auc(y_true, y_prob),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "drusen_to_cnv": int(cm[DRUSEN_CLASS, 0]),
        "drusen_to_normal": int(cm[DRUSEN_CLASS, NORMAL_CLASS]),
        "normal_to_drusen": int(cm[NORMAL_CLASS, DRUSEN_CLASS]),
    }
    for class_id, class_name in enumerate(CLASS_NAMES):
        key = class_name.lower()
        metrics[f"{key}_precision"] = per_precision[class_id]
        metrics[f"{key}_recall"] = per_recall[class_id]
        metrics[f"{key}_f1"] = per_f1[class_id]
    return metrics, cm


def evaluate_loss(
    model,
    loader,
    criterion,
    *,
    use_amp: bool = EVALUATION_USE_AMP,
):
    """Calcula a loss de validação em float32 por padrão."""
    model.eval()
    total, batches = 0.0, 0

    with torch.no_grad():
        for batch_number, (images, labels, _) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(
                device_type=device.type,
                enabled=bool(use_amp and AMP_ENABLED),
            ):
                logits = model(images)

            logits = logits.float()

            if not torch.isfinite(logits).all():
                raise FloatingPointError(
                    "Logits contêm NaN ou infinito ao calcular "
                    f"val_loss no lote {batch_number}."
                )

            loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "A loss de validação tornou-se NaN ou infinita "
                    f"no lote {batch_number}."
                )

            total += float(loss.item())
            batches += 1

    return total / max(1, batches)


In [14]:
def train_model_run(
    model_name: str,
    strategy: str,
    seed: int,
    learning_rate: float,
    weight_decay: float,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    train_indices: Optional[np.ndarray] = None,
    progress_context: Optional[Dict[str, Any]] = None,
    training_amp_enabled: Optional[bool] = None,
):
    seed_everything(seed)
    spec = MODEL_SPECS[model_name]
    start = time.perf_counter()

    run_amp_enabled = (
        AMP_ENABLED
        if training_amp_enabled is None
        else bool(training_amp_enabled and AMP_ENABLED)
    )

    context = progress_context or {}
    phase = context.get("phase", "Treinamento")
    run_index = int(context.get("run_index", 1))
    run_total = max(1, int(context.get("run_total", 1)))

    TRAINING_MONITOR.update(
        phase=phase,
        model_name=model_name,
        strategy=strategy,
        seed=seed,
        run_index=run_index,
        run_total=run_total,
        epoch=0,
        max_epochs=max_epochs,
        status="Preparando dados e modelo",
        patience=patience,
        start_time=start,
    )

    train_loader, val_loader, test_loader = make_loaders(
        image_size=spec["input_size"],
        seed=seed,
        physical_batch_size=spec["physical_batch_size"],
        strategy=strategy,
        train_indices=train_indices,
    )

    model = spec["builder"]().to(device)
    criterion = nn.CrossEntropyLoss(
        weight=class_loss_weights() if strategy == "class_weighted" else None
    )
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs)
    scaler = GradScaler("cuda", enabled=run_amp_enabled)

    best_auc = -np.inf
    best_epoch = -1
    best_state = None
    no_improvement = 0
    history_rows = []
    accumulation_steps = spec["accumulation_steps"]
    last_val_loss = None
    last_val_auc = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, batches = 0.0, 0
        last_dashboard_update = time.perf_counter()

        TRAINING_MONITOR.update(
            phase=phase,
            model_name=model_name,
            strategy=strategy,
            seed=seed,
            run_index=run_index,
            run_total=run_total,
            epoch=epoch,
            max_epochs=max_epochs,
            status="Treinando",
            batch_index=0,
            batch_total=len(train_loader),
            train_loss=None,
            val_loss=last_val_loss,
            val_auc=last_val_auc,
            best_auc=best_auc,
            best_epoch=best_epoch,
            no_improvement=no_improvement,
            patience=patience,
            start_time=start,
        )

        for batch_index, (images, labels, _) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(
                device_type=device.type,
                enabled=run_amp_enabled,
            ):
                logits = model(images)
                loss = criterion(logits, labels) / accumulation_steps

            if not torch.isfinite(logits).all():
                raise FloatingPointError(
                    "Logits não finitos durante treinamento: "
                    f"modelo={model_name}, estratégia={strategy}, "
                    f"seed={seed}, época={epoch}, lote={batch_index}, "
                    f"AMP={run_amp_enabled}."
                )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Loss não finita durante treinamento: "
                    f"modelo={model_name}, estratégia={strategy}, "
                    f"seed={seed}, época={epoch}, lote={batch_index}, "
                    f"AMP={run_amp_enabled}."
                )

            scaler.scale(loss).backward()

            if (
                batch_index % accumulation_steps == 0
                or batch_index == len(train_loader)
            ):
                scaler.unscale_(optimizer)
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=MAX_GRAD_NORM,
                )

                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Norma do gradiente não finita: "
                        f"modelo={model_name}, estratégia={strategy}, "
                        f"seed={seed}, época={epoch}, lote={batch_index}, "
                        f"AMP={run_amp_enabled}."
                    )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            total_loss += float(loss.item()) * accumulation_steps
            batches += 1

            now = time.perf_counter()
            should_refresh = (
                now - last_dashboard_update >= DASHBOARD_UPDATE_SECONDS
                or batch_index == len(train_loader)
            )
            if should_refresh:
                TRAINING_MONITOR.update(
                    phase=phase,
                    model_name=model_name,
                    strategy=strategy,
                    seed=seed,
                    run_index=run_index,
                    run_total=run_total,
                    epoch=epoch,
                    max_epochs=max_epochs,
                    status="Treinando",
                    batch_index=batch_index,
                    batch_total=len(train_loader),
                    train_loss=total_loss / max(1, batches),
                    val_loss=last_val_loss,
                    val_auc=last_val_auc,
                    best_auc=best_auc,
                    best_epoch=best_epoch,
                    no_improvement=no_improvement,
                    patience=patience,
                    start_time=start,
                )
                last_dashboard_update = now

        scheduler.step()
        train_loss = total_loss / max(1, batches)

        TRAINING_MONITOR.update(
            phase=phase,
            model_name=model_name,
            strategy=strategy,
            seed=seed,
            run_index=run_index,
            run_total=run_total,
            epoch=epoch,
            max_epochs=max_epochs,
            status="Validando",
            train_loss=train_loss,
            val_loss=last_val_loss,
            val_auc=last_val_auc,
            best_auc=best_auc,
            best_epoch=best_epoch,
            no_improvement=no_improvement,
            patience=patience,
            start_time=start,
        )

        val_loss = evaluate_loss(model, val_loader, criterion)
        _, val_true, val_prob, _ = collect_predictions(model, val_loader)
        val_auc = compute_macro_auc(val_true, val_prob)
        last_val_loss = val_loss
        last_val_auc = val_auc

        history_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_auc": val_auc,
            "learning_rate": optimizer.param_groups[0]["lr"],
        })

        if val_auc > best_auc + MIN_DELTA:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            no_improvement = 0
        else:
            no_improvement += 1

        TRAINING_MONITOR.update(
            phase=phase,
            model_name=model_name,
            strategy=strategy,
            seed=seed,
            run_index=run_index,
            run_total=run_total,
            epoch=epoch,
            max_epochs=max_epochs,
            status="Época concluída",
            train_loss=train_loss,
            val_loss=val_loss,
            val_auc=val_auc,
            best_auc=best_auc,
            best_epoch=best_epoch,
            no_improvement=no_improvement,
            patience=patience,
            start_time=start,
        )

        if no_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("Nenhum checkpoint foi selecionado.")

    stop_epoch = history_rows[-1]["epoch"]
    model.load_state_dict(best_state)

    TRAINING_MONITOR.update(
        phase=phase,
        model_name=model_name,
        strategy=strategy,
        seed=seed,
        run_index=run_index,
        run_total=run_total,
        epoch=stop_epoch,
        max_epochs=max_epochs,
        status="Avaliando conjunto de teste",
        train_loss=history_rows[-1]["train_loss"],
        val_loss=history_rows[-1]["val_loss"],
        val_auc=history_rows[-1]["val_macro_auc"],
        best_auc=best_auc,
        best_epoch=best_epoch,
        no_improvement=no_improvement,
        patience=patience,
        start_time=start,
    )

    test_indices, y_true, y_prob, y_pred = collect_predictions(model, test_loader)
    metrics, cm = compute_metrics(y_true, y_prob)

    row = {
        "model": model_name,
        "strategy": strategy,
        "seed": seed,
        "input_size": spec["input_size"],
        "physical_batch_size": spec["physical_batch_size"],
        "effective_batch_size": (
            spec["physical_batch_size"] * spec["accumulation_steps"]
        ),
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "loss": (
            "class-weighted CrossEntropyLoss"
            if strategy == "class_weighted"
            else "CrossEntropyLoss"
        ),
        "pretrained": False,
        "training_amp_enabled": run_amp_enabled,
        "max_grad_norm": MAX_GRAD_NORM,
        "checkpoint_criterion": "validation macro-AUC",
        "best_val_macro_auc": best_auc,
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
        "trainable_parameters": count_parameters(model),
        "runtime_seconds": time.perf_counter() - start,
        **metrics,
    }

    history_df = pd.DataFrame(history_rows)
    predictions_df = pd.DataFrame({
        "test_index": test_indices,
        "y_true": y_true,
        "y_pred": y_pred,
        **{f"prob_{CLASS_NAMES[i].lower()}": y_prob[:, i] for i in range(NUM_CLASSES)},
    })

    TRAINING_MONITOR.update(
        phase=phase,
        model_name=model_name,
        strategy=strategy,
        seed=seed,
        run_index=run_index,
        run_total=run_total,
        epoch=stop_epoch,
        max_epochs=stop_epoch,
        status="Concluído",
        train_loss=history_rows[-1]["train_loss"],
        val_loss=history_rows[-1]["val_loss"],
        val_auc=history_rows[-1]["val_macro_auc"],
        best_auc=best_auc,
        best_epoch=best_epoch,
        no_improvement=no_improvement,
        patience=patience,
        start_time=start,
        final_metrics=metrics,
    )

    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, history_df, predictions_df, cm


## 5. Busca de hiperparâmetros com reutilização após reinício

Quando `selected_hyperparameters.csv` já existe e contém todos os modelos, o notebook reutiliza o arquivo e não repete a busca.


In [15]:
def proxy_indices(targets: np.ndarray, fraction: float, seed: int = 2026):
    if fraction >= 1.0:
        return np.arange(len(targets))
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=fraction,
        random_state=seed,
    )
    indices, _ = next(splitter.split(np.zeros(len(targets)), targets))
    return np.sort(indices)


TUNING_INDICES = proxy_indices(TRAIN_TARGETS, TUNING_FRACTION)
TUNING_SEED = 2026
TUNING_RESULTS_PATH = RESULTS_DIR / "hyperparameter_tuning_trials.csv"
SELECTED_HYPERPARAMETERS_PATH = RESULTS_DIR / "selected_hyperparameters.csv"

TUNING_COMBINATIONS = list(itertools.product(
    TUNING_LEARNING_RATES,
    TUNING_WEIGHT_DECAYS,
))
TUNING_PROGRESS = {
    "current": 0,
    "total": max(1, len(MODELS_TO_RUN) * len(TUNING_COMBINATIONS)),
}


def tune_model(model_name: str):
    trial_rows = []
    best = None

    for learning_rate, weight_decay in TUNING_COMBINATIONS:
        TUNING_PROGRESS["current"] += 1

        row, _, _, _ = train_model_run(
            model_name=model_name,
            strategy="natural",
            seed=TUNING_SEED,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            max_epochs=TUNING_EPOCHS,
            patience=TUNING_EPOCHS,
            train_indices=TUNING_INDICES,
            progress_context={
                "phase": "Busca de hiperparâmetros",
                "run_index": TUNING_PROGRESS["current"],
                "run_total": TUNING_PROGRESS["total"],
            },
        )

        trial = {
            "model": model_name,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "proxy_val_macro_auc": row["best_val_macro_auc"],
            "runtime_seconds": row["runtime_seconds"],
        }
        trial_rows.append(trial)

        if best is None or trial["proxy_val_macro_auc"] > best["proxy_val_macro_auc"]:
            best = trial

    return pd.DataFrame(trial_rows), best


def selected_hyperparameters_are_complete(frame: pd.DataFrame) -> bool:
    required_columns = {
        "model",
        "learning_rate",
        "weight_decay",
    }

    if frame.empty or not required_columns.issubset(frame.columns):
        return False

    saved_models = set(frame["model"].astype(str))
    required_models = set(map(str, MODELS_TO_RUN))
    return required_models.issubset(saved_models)


# Após reiniciar o kernel, não refaz a busca se o arquivo completo já existe.
selected_hyperparameters = pd.DataFrame()

if SELECTED_HYPERPARAMETERS_PATH.exists():
    try:
        saved_hyperparameters = pd.read_csv(SELECTED_HYPERPARAMETERS_PATH)

        if selected_hyperparameters_are_complete(saved_hyperparameters):
            selected_hyperparameters = (
                saved_hyperparameters
                .loc[
                    saved_hyperparameters["model"].astype(str).isin(
                        set(map(str, MODELS_TO_RUN))
                    )
                ]
                .drop_duplicates(subset=["model"], keep="last")
                .reset_index(drop=True)
            )

            print(
                "Busca de hiperparâmetros já concluída. "
                "Arquivo salvo reutilizado:"
            )
            print(SELECTED_HYPERPARAMETERS_PATH)

    except (pd.errors.EmptyDataError, OSError, ValueError) as error:
        print(
            "Não foi possível reutilizar selected_hyperparameters.csv. "
            f"A busca será executada novamente. Motivo: {error}"
        )


if selected_hyperparameters.empty:
    if RUN_HYPERPARAMETER_TUNING:
        all_trials = []
        selected_rows = []

        for model_name in MODELS_TO_RUN:
            trials, best = tune_model(model_name)
            all_trials.append(trials)
            selected_rows.append(best)

        tuning_results = pd.concat(all_trials, ignore_index=True)
        selected_hyperparameters = pd.DataFrame(selected_rows)

        tuning_results.to_csv(
            TUNING_RESULTS_PATH,
            index=False,
        )
        selected_hyperparameters.to_csv(
            SELECTED_HYPERPARAMETERS_PATH,
            index=False,
        )
    else:
        selected_hyperparameters = pd.DataFrame([
            {
                "model": model_name,
                "learning_rate": DEFAULT_LR,
                "weight_decay": DEFAULT_WEIGHT_DECAY,
                "proxy_val_macro_auc": np.nan,
                "runtime_seconds": np.nan,
            }
            for model_name in MODELS_TO_RUN
        ])
        selected_hyperparameters.to_csv(
            SELECTED_HYPERPARAMETERS_PATH,
            index=False,
        )

display(selected_hyperparameters)


Busca de hiperparâmetros já concluída. Arquivo salvo reutilizado:
results_baselines_revised_final/selected_hyperparameters.csv


,model,learning_rate,weight_decay,proxy_val_macro_auc,runtime_seconds
0,ResNet18,0.0003,0.00010,0.972600,90.277755
1,DenseNet121,0.0003,0.00001,0.974981,486.522304
2,EfficientNet-B0-CBAM-LT,0.0010,0.00010,0.968749,111.593357
3,MedConvMixer-LT,0.0010,0.00010,0.965562,49.963150
4,TinyViT-LT,0.0003,0.00001,0.937041,106.268513
5,MedMobileViT-LT,0.0007,0.00010,0.968622,167.871861
6,MaxViT-LT,0.0003,0.00001,0.973951,1637.651464
7,Swin-Transformer-LT,0.0003,0.00010,0.619399,752.182466


## 6. Execuções finais com retomada por `raw_results.csv`

- combinações concluídas e registradas no CSV são puladas;
- a combinação interrompida, sem linha no CSV, reinicia da época 1;
- validação e softmax são calculados em `float32` para evitar NaN;
- o treinamento usa clipping de gradiente;
- se houver instabilidade com AMP, apenas a combinação pendente é reiniciada sem AMP;
- os artefatos são salvos antes do registro definitivo no CSV;
- a escrita do CSV é atômica para reduzir risco de corrupção.


In [16]:
RAW_RESULTS_PATH = RESULTS_DIR / "raw_results.csv"
RUN_KEY_COLUMNS = ["model", "strategy", "seed"]


def atomic_save_csv(frame: pd.DataFrame, path: Path) -> None:
    """
    Salva primeiro em arquivo temporário e só depois substitui o CSV oficial.

    Isso reduz o risco de raw_results.csv ficar incompleto se o kernel
    interromper exatamente durante a escrita.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")

    frame.to_csv(
        temporary_path,
        index=False,
    )
    os.replace(
        temporary_path,
        path,
    )


def normalize_run_key(
    model_name: str,
    strategy: str,
    seed: int,
) -> Tuple[str, str, int]:
    return (
        str(model_name),
        str(strategy),
        int(seed),
    )


def load_raw_results(path: Path) -> pd.DataFrame:
    """
    Carrega resultados concluídos.

    Uma combinação só é considerada concluída quando possui uma linha
    registrada em raw_results.csv.
    """
    if not path.exists():
        return pd.DataFrame()

    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

    if frame.empty:
        return frame

    missing_columns = [
        column
        for column in RUN_KEY_COLUMNS
        if column not in frame.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "raw_results.csv existe, mas não contém as colunas necessárias: "
            + ", ".join(missing_columns)
        )

    frame["model"] = frame["model"].astype(str)
    frame["strategy"] = frame["strategy"].astype(str)
    frame["seed"] = pd.to_numeric(
        frame["seed"],
        errors="raise",
    ).astype(int)

    # Se uma execução foi registrada mais de uma vez, mantém a última.
    frame = (
        frame
        .drop_duplicates(
            subset=RUN_KEY_COLUMNS,
            keep="last",
        )
        .reset_index(drop=True)
    )

    return frame


raw_results = load_raw_results(RAW_RESULTS_PATH)

completed = set()
if not raw_results.empty:
    completed = {
        normalize_run_key(model_name, strategy, seed)
        for model_name, strategy, seed in raw_results[RUN_KEY_COLUMNS].itertuples(
            index=False,
            name=None,
        )
    }

all_run_keys = [
    normalize_run_key(model_name, strategy, seed)
    for model_name in MODELS_TO_RUN
    for strategy in STRATEGIES_TO_RUN
    for seed in SEEDS
]

total_runs = len(all_run_keys)
completed_in_plan = sum(
    key in completed
    for key in all_run_keys
)
run_counter = completed_in_plan

print("=" * 72)
print("RETOMADA DOS EXPERIMENTOS")
print("=" * 72)
print(f"Execuções previstas: {total_runs}")
print(f"Já concluídas no raw_results.csv: {completed_in_plan}")
print(f"Pendentes: {total_runs - completed_in_plan}")
print(
    "Regra: combinações registradas serão puladas; "
    "a combinação interrompida será reiniciada desde a época 1."
)
print("=" * 72)

hyperparameter_lookup = (
    selected_hyperparameters
    .drop_duplicates(subset=["model"], keep="last")
    .set_index("model")
)

for model_name in MODELS_TO_RUN:
    if model_name not in hyperparameter_lookup.index:
        raise RuntimeError(
            f"Não há hiperparâmetros selecionados para o modelo {model_name}."
        )

    learning_rate = float(
        hyperparameter_lookup.loc[
            model_name,
            "learning_rate",
        ]
    )
    weight_decay = float(
        hyperparameter_lookup.loc[
            model_name,
            "weight_decay",
        ]
    )

    for strategy in STRATEGIES_TO_RUN:
        for seed in SEEDS:
            key = normalize_run_key(
                model_name,
                strategy,
                seed,
            )

            safe_model = (
                model_name
                .replace("/", "_")
                .replace(" ", "_")
            )
            safe_name = (
                f"{safe_model}_{strategy}_seed{seed}"
            )

            history_path = (
                RESULTS_DIR
                / "histories"
                / f"{safe_name}.csv"
            )
            predictions_path = (
                RESULTS_DIR
                / "predictions"
                / f"{safe_name}.csv"
            )
            confusion_matrix_path = (
                RESULTS_DIR
                / "confusion_matrices"
                / f"{safe_name}.npy"
            )

            # ---------------------------------------------------------
            # CASO 1: execução concluída anteriormente.
            # ---------------------------------------------------------
            if key in completed:
                print(
                    "[PULADO] "
                    f"modelo={model_name} | "
                    f"estratégia={strategy} | "
                    f"seed={seed} "
                    "já está registrado em raw_results.csv."
                )
                continue

            # ---------------------------------------------------------
            # CASO 2: não existe linha no CSV.
            #
            # Isso inclui a combinação que estava treinando quando o
            # kernel caiu. Ela será executada novamente desde a época 1.
            # ---------------------------------------------------------
            run_counter += 1

            print()
            print("-" * 72)
            print(
                "[INICIANDO OU REINICIANDO DO ZERO] "
                f"execução {run_counter}/{total_runs}"
            )
            print(
                f"modelo={model_name} | "
                f"estratégia={strategy} | "
                f"seed={seed}"
            )
            print(
                "Motivo: a combinação ainda não possui linha "
                "em raw_results.csv."
            )
            print("-" * 72)

            try:
                row, history_df, predictions_df, cm = train_model_run(
                    model_name=model_name,
                    strategy=strategy,
                    seed=seed,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    progress_context={
                        "phase": "Experimentos finais",
                        "run_index": run_counter,
                        "run_total": total_runs,
                    },
                    training_amp_enabled=AMP_ENABLED,
                )
                row["numerical_retry_without_amp"] = False

            except FloatingPointError as error:
                if not RETRY_WITHOUT_AMP_ON_NAN or not AMP_ENABLED:
                    raise

                print()
                print("=" * 72)
                print("[INSTABILIDADE NUMÉRICA DETECTADA]")
                print(error)
                print(
                    "A combinação pendente será reiniciada da época 1 "
                    "com treinamento em float32 e o mesmo learning rate."
                )
                print("=" * 72)

                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                row, history_df, predictions_df, cm = train_model_run(
                    model_name=model_name,
                    strategy=strategy,
                    seed=seed,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    progress_context={
                        "phase": "Reexecução estável sem AMP",
                        "run_index": run_counter,
                        "run_total": total_runs,
                    },
                    training_amp_enabled=False,
                )
                row["numerical_retry_without_amp"] = True

            # Primeiro salva os artefatos completos da execução.
            history_df.to_csv(
                history_path,
                index=False,
            )
            predictions_df.to_csv(
                predictions_path,
                index=False,
            )
            np.save(
                confusion_matrix_path,
                cm,
            )

            # Só depois registra a execução como concluída.
            completed_row = pd.DataFrame([row])

            raw_results = pd.concat(
                [
                    raw_results,
                    completed_row,
                ],
                ignore_index=True,
            )

            raw_results["model"] = raw_results["model"].astype(str)
            raw_results["strategy"] = raw_results["strategy"].astype(str)
            raw_results["seed"] = pd.to_numeric(
                raw_results["seed"],
                errors="raise",
            ).astype(int)

            raw_results = (
                raw_results
                .drop_duplicates(
                    subset=RUN_KEY_COLUMNS,
                    keep="last",
                )
                .sort_values(
                    RUN_KEY_COLUMNS,
                    kind="stable",
                )
                .reset_index(drop=True)
            )

            atomic_save_csv(
                raw_results,
                RAW_RESULTS_PATH,
            )

            # Atualiza o conjunto em memória para impedir repetição
            # durante a mesma execução do notebook.
            completed.add(key)

            print(
                "[CONCLUÍDO E REGISTRADO] "
                f"modelo={model_name} | "
                f"estratégia={strategy} | "
                f"seed={seed}"
            )
            print(f"Histórico: {history_path}")
            print(f"Predições: {predictions_path}")
            print(f"Matriz de confusão: {confusion_matrix_path}")
            print(f"Registro principal: {RAW_RESULTS_PATH}")


raw_results = load_raw_results(RAW_RESULTS_PATH)

print()
print("=" * 72)
print(
    f"Execuções registradas: "
    f"{len(raw_results)}/{total_runs}"
)
print("=" * 72)

summary_columns = [
    "model",
    "strategy",
    "seed",
    "accuracy",
    "macro_auc",
    "macro_f1",
    "runtime_seconds",
]

if raw_results.empty:
    print("Nenhuma execução foi concluída.")
else:
    display(
        raw_results[summary_columns]
        .tail(min(10, len(raw_results)))
    )


RETOMADA DOS EXPERIMENTOS
Execuções previstas: 160
Já concluídas no raw_results.csv: 121
Pendentes: 39
Regra: combinações registradas serão puladas; a combinação interrompida será reiniciada desde a época 1.
[PULADO] modelo=ResNet18 | estratégia=natural | seed=42 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=natural | seed=123 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=natural | seed=456 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=natural | seed=789 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=natural | seed=2026 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=cws | seed=42 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=cws | seed=123 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia=cws | seed=456 já está registrado em raw_results.csv.
[PULADO] modelo=ResNet18 | estratégia


[INSTABILIDADE NUMÉRICA DETECTADA]
Norma do gradiente não finita: modelo=MaxViT-LT, estratégia=natural, seed=123, época=1, lote=164, AMP=True.
A combinação pendente será reiniciada da época 1 com treinamento em float32 e o mesmo learning rate.
[CONCLUÍDO E REGISTRADO] modelo=MaxViT-LT | estratégia=natural | seed=123
Histórico: results_baselines_revised_final/histories/MaxViT-LT_natural_seed123.csv
Predições: results_baselines_revised_final/predictions/MaxViT-LT_natural_seed123.csv
Matriz de confusão: results_baselines_revised_final/confusion_matrices/MaxViT-LT_natural_seed123.npy
Registro principal: results_baselines_revised_final/raw_results.csv

------------------------------------------------------------------------
[INICIANDO OU REINICIANDO DO ZERO] execução 123/160
modelo=MaxViT-LT | estratégia=natural | seed=456
Motivo: a combinação ainda não possui linha em raw_results.csv.
------------------------------------------------------------------------

[INSTABILIDADE NUMÉRICA DETECTA

KeyboardInterrupt: 

## 7. Tabelas, curvas e matrizes

A tabela geral usa somente macro-AUC de teste. `best_val_macro_auc` permanece em arquivo separado para auditoria do checkpoint.


In [17]:
METRICS = [
    "accuracy",
    "macro_auc",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "mcc",
    "drusen_recall",
    "normal_recall",
    "drusen_to_cnv",
    "normal_to_drusen",
    "runtime_seconds",
]

aggregate_rows = []
for (model_name, strategy), group in raw_results.groupby(
    ["model", "strategy"],
    sort=False,
):
    row = {
        "model": model_name,
        "strategy": strategy,
        "n_runs": len(group),
        "learning_rate": group["learning_rate"].iloc[0],
        "weight_decay": group["weight_decay"].iloc[0],
        "trainable_parameters": group["trainable_parameters"].iloc[0],
    }
    for metric in METRICS:
        mean = group[metric].mean()
        std = group[metric].std(ddof=1) if len(group) > 1 else np.nan
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
        row[f"{metric}_formatted"] = (
            f"{mean:.4f} ± {std:.4f}"
            if len(group) > 1
            else f"{mean:.4f}"
        )
    aggregate_rows.append(row)

aggregate = pd.DataFrame(aggregate_rows).sort_values(
    ["strategy", "macro_f1_mean"],
    ascending=[True, False],
)
display(aggregate[[
    "model",
    "strategy",
    "n_runs",
    "accuracy_formatted",
    "macro_auc_formatted",
    "macro_f1_formatted",
    "drusen_recall_formatted",
    "normal_recall_formatted",
]])
aggregate.to_csv(RESULTS_DIR / "aggregate_results.csv", index=False)

natural_table = aggregate[aggregate["strategy"] == "natural"].copy()
natural_table.to_csv(RESULTS_DIR / "main_natural_protocol_table.csv", index=False)


,model,strategy,n_runs,accuracy_formatted,macro_auc_formatted,macro_f1_formatted,drusen_recall_formatted,normal_recall_formatted
13,MedMobileViT-LT,balanced_sampler,5,0.7992 ± 0.0096,0.9618 ± 0.0033,0.7883 ± 0.0132,0.4832 ± 0.0487,0.8616 ± 0.0410
4,EfficientNet-B0-CBAM-LT,balanced_sampler,5,0.7988 ± 0.0113,0.9610 ± 0.0029,0.7847 ± 0.0144,0.4504 ± 0.0487,0.8656 ± 0.0197
9,MedConvMixer-LT,balanced_sampler,5,0.7922 ± 0.0055,0.9598 ± 0.0015,0.7806 ± 0.0055,0.4760 ± 0.0242,0.8440 ± 0.0144
0,DenseNet121,balanced_sampler,5,0.7980 ± 0.0170,0.9665 ± 0.0007,0.7777 ± 0.0210,0.3816 ± 0.0621,0.9432 ± 0.0296
17,ResNet18,balanced_sampler,5,0.7862 ± 0.0166,0.9577 ± 0.0061,0.7659 ± 0.0218,0.3712 ± 0.0610,0.9264 ± 0.0140
21,TinyViT-LT,balanced_sampler,5,0.7660 ± 0.0241,0.9478 ± 0.0064,0.7443 ± 0.0308,0.3624 ± 0.0702,0.9160 ± 0.0330
14,MedMobileViT-LT,class_weighted,5,0.8164 ± 0.0078,0.9662 ± 0.0034,0.8051 ± 0.0081,0.4856 ± 0.0166,0.9064 ± 0.0119
10,MedConvMixer-LT,class_weighted,5,0.8082 ± 0.0070,0.9628 ± 0.0030,0.7987 ± 0.0078,0.5096 ± 0.0187,0.8648 ± 0.0148
1,DenseNet121,class_weighted,5,0.8014 ± 0.0113,0.9640 ± 0.0016,0.7867 ± 0.0131,0.4336 ± 0.0347,0.9168 ± 0.0191
5,EfficientNet-B0-CBAM-LT,class_weighted,5,0.7984 ± 0.0112,0.9610 ± 0.0034,0.7853 ± 0.0113,0.4592 ± 0.0225,0.8608 ± 0.0299


In [19]:
# Uma figura compacta para comparar a evolução da macro-AUC de validação.
representative_seed = SEEDS[0]
plt.figure(figsize=(10, 6))

for model_name in MODELS_TO_RUN:
    safe_model = model_name.replace("/", "_").replace(" ", "_")
    history_path = (
        RESULTS_DIR
        / "histories"
        / f"{safe_model}_natural_seed{representative_seed}.csv"
    )
    if history_path.exists():
        history = pd.read_csv(history_path)
        plt.plot(
            history["epoch"],
            history["val_macro_auc"],
            label=model_name,
        )

plt.xlabel("Epoch")
plt.ylabel("Validation macro-AUC")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "figures" / "all_models_validation_auc.pdf",
    bbox_inches="tight",
)
plt.show() if SHOW_FIGURES else plt.close()


In [20]:
# Curvas de treino e validação para todos os modelos, uma figura por modelo.
for model_name in MODELS_TO_RUN:
    safe_model = model_name.replace("/", "_").replace(" ", "_")
    history_path = (
        RESULTS_DIR
        / "histories"
        / f"{safe_model}_natural_seed{representative_seed}.csv"
    )
    if not history_path.exists():
        continue

    history = pd.read_csv(history_path)
    plt.figure(figsize=(7, 4))
    plt.plot(history["epoch"], history["train_loss"], label="Train loss")
    plt.plot(history["epoch"], history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(model_name)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / "figures" / f"{safe_model}_loss.pdf",
        bbox_inches="tight",
    )
    plt.show() if SHOW_FIGURES else plt.close()


In [21]:
# Matrizes de confusão para todos os modelos no protocolo natural.
for model_name in MODELS_TO_RUN:
    safe_model = model_name.replace("/", "_").replace(" ", "_")
    cm_path = (
        RESULTS_DIR
        / "confusion_matrices"
        / f"{safe_model}_natural_seed{representative_seed}.npy"
    )
    if not cm_path.exists():
        continue

    cm = np.load(cm_path)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45)
    plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(model_name)

    for row_index in range(NUM_CLASSES):
        for column_index in range(NUM_CLASSES):
            plt.text(
                column_index,
                row_index,
                str(cm[row_index, column_index]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / "figures" / f"{safe_model}_confusion_matrix.pdf",
        bbox_inches="tight",
    )
    plt.show() if SHOW_FIGURES else plt.close()


In [22]:
# Efeito de cada estratégia em Drusen e Normal.
plt.figure(figsize=(9, 6))
for (model_name, strategy), group in raw_results.groupby(["model", "strategy"]):
    plt.scatter(
        group["drusen_recall"].mean(),
        group["normal_recall"].mean(),
        label=f"{model_name} | {strategy}",
    )
plt.xlabel("Drusen recall")
plt.ylabel("Normal recall")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "figures" / "drusen_normal_tradeoff_all_models.pdf",
    bbox_inches="tight",
)
plt.show() if SHOW_FIGURES else plt.close()


In [23]:
latex_columns = [
    "model",
    "strategy",
    "accuracy_formatted",
    "macro_auc_formatted",
    "macro_f1_formatted",
    "mcc_formatted",
    "drusen_recall_formatted",
    "normal_recall_formatted",
]

latex_table = aggregate[latex_columns].rename(columns={
    "model": "Model",
    "strategy": "Strategy",
    "accuracy_formatted": "ACC",
    "macro_auc_formatted": "AUC",
    "macro_f1_formatted": "Macro-F1",
    "mcc_formatted": "MCC",
    "drusen_recall_formatted": "Drusen REC",
    "normal_recall_formatted": "Normal REC",
})

latex_text = latex_table.to_latex(index=False, escape=False)
print(latex_text)
with open(RESULTS_DIR / "baseline_results_table.tex", "w", encoding="utf-8") as file:
    file.write(latex_text)

protocol = {
    "mode": MODE,
    "seeds": SEEDS,
    "models": MODELS_TO_RUN,
    "strategies": STRATEGIES_TO_RUN,
    "dataset": "OCTMNIST official splits",
    "pretrained": False,
    "loss": "CrossEntropyLoss, with inverse-frequency weights only in class_weighted",
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "checkpoint_criterion": "validation macro-AUC",
    "test_metric_policy": "all main-table AUC values are test macro-AUC",
    "augmentation_policy": "same transform distribution for every class",
    "hyperparameter_tuning": {
        "enabled": RUN_HYPERPARAMETER_TUNING,
        "learning_rates": TUNING_LEARNING_RATES,
        "weight_decays": TUNING_WEIGHT_DECAYS,
        "proxy_epochs": TUNING_EPOCHS,
        "selection": "validation macro-AUC",
        "same_budget_for_all_models": True,
    },
}
with open(RESULTS_DIR / "training_protocol.json", "w", encoding="utf-8") as file:
    json.dump(protocol, file, ensure_ascii=False, indent=2)


\begin{tabular}{llllllll}
\toprule
Model & Strategy & ACC & AUC & Macro-F1 & MCC & Drusen REC & Normal REC \\
\midrule
MedMobileViT-LT & balanced_sampler & 0.7992 ± 0.0096 & 0.9618 ± 0.0033 & 0.7883 ± 0.0132 & 0.7442 ± 0.0118 & 0.4832 ± 0.0487 & 0.8616 ± 0.0410 \\
EfficientNet-B0-CBAM-LT & balanced_sampler & 0.7988 ± 0.0113 & 0.9610 ± 0.0029 & 0.7847 ± 0.0144 & 0.7447 ± 0.0130 & 0.4504 ± 0.0487 & 0.8656 ± 0.0197 \\
MedConvMixer-LT & balanced_sampler & 0.7922 ± 0.0055 & 0.9598 ± 0.0015 & 0.7806 ± 0.0055 & 0.7336 ± 0.0075 & 0.4760 ± 0.0242 & 0.8440 ± 0.0144 \\
DenseNet121 & balanced_sampler & 0.7980 ± 0.0170 & 0.9665 ± 0.0007 & 0.7777 ± 0.0210 & 0.7497 ± 0.0177 & 0.3816 ± 0.0621 & 0.9432 ± 0.0296 \\
ResNet18 & balanced_sampler & 0.7862 ± 0.0166 & 0.9577 ± 0.0061 & 0.7659 ± 0.0218 & 0.7347 ± 0.0187 & 0.3712 ± 0.0610 & 0.9264 ± 0.0140 \\
TinyViT-LT & balanced_sampler & 0.7660 ± 0.0241 & 0.9478 ± 0.0064 & 0.7443 ± 0.0308 & 0.7045 ± 0.0293 & 0.3624 ± 0.0702 & 0.9160 ± 0.0330 \\
MedMobileViT-